## 📌 Carga de datos y construcción del dataset final

**Objetivo:**  
Construir el dataset base para el modelado a partir de los datos crudos, aplicando transformaciones y generando variables relevantes.

**Descripción:**  
En este bloque se carga el dataset original desde un archivo CSV previamente obtenido. Luego, se aplican funciones de preprocesamiento definidas en un módulo externo (`src/preprocessing.py`) con el fin de enriquecer los datos y dejarlos listos para el entrenamiento de modelos.

Las transformaciones realizadas incluyen:

- **Incorporación de variables temporales:**
  - Año (`YEAR`)
  - Mes (`MONTH`)
  - Día (`DAY`)

- **Generación de variables climáticas derivadas:**
  - Temperatura media (`TEMP_MEAN`)
  - Rango de temperatura (`TEMP_RANGE`)

- **Creación de variables objetivo (targets):**
  - Indicador de lluvia (`RAIN`)
  - Indicador de nieve (`SNOW_DAY`)

Finalmente, el dataset se ordena cronológicamente por fecha para preservar la estructura temporal de los datos.

**Resultados / Insights:**
- Se obtiene un dataset enriquecido y estructurado, listo para ser utilizado en tareas de modelado.
- La incorporación de variables derivadas permite capturar mejor la dinámica climática.
- El orden temporal resulta clave para evitar fugas de información en etapas posteriores (train/test split).

In [ ]:
import sys
import os
import pandas as pd

sys.path.append(os.path.abspath("../src"))

from preprocessing import (
    add_time_features,
    add_temperature_features,
    create_rain_target,
    create_snow_target
)

df = pd.read_csv("../data/raw/ghcn_daily_data.csv", parse_dates=["DATE"])

# Preprocesamiento de los datos
df = add_time_features(df)
df = add_temperature_features(df)
df = create_rain_target(df)
df = create_snow_target(df)
df = df.sort_values("DATE")

## 📌 Separación de datos y definición de variables

**Objetivo:**  
Dividir el dataset en conjuntos de entrenamiento y prueba respetando la estructura temporal, y definir las variables predictoras y la variable objetivo.

**Descripción:**  
Se realiza una partición del dataset basada en el año (`YEAR`), utilizando los datos anteriores a 2025 como conjunto de entrenamiento y los datos de 2025 en adelante como conjunto de prueba.

Este enfoque evita la fuga de información (*data leakage*) y simula un escenario real donde se entrena el modelo con datos históricos para predecir eventos futuros.

Luego, se definen:

- **Variables predictoras (features):**
  - Temperatura máxima (`TMAX`)
  - Temperatura mínima (`TMIN`)
  - Temperatura media (`TEMP_MEAN`)
  - Rango térmico (`TEMP_RANGE`)
  - Nieve (`SNOW`)

- **Variable objetivo (target):**
  - Indicador de lluvia (`RAIN`)

Finalmente, se construyen las matrices de entrada (`X_train`, `X_test`) y las variables objetivo (`y_train`, `y_test`) para el modelado.

**Resultados / Insights:**
- Se preserva la coherencia temporal del dataset.
- Se definen variables relevantes basadas en el análisis exploratorio previo.
- El dataset queda preparado para aplicar técnicas de selección de variables y entrenamiento de modelos.

In [ ]:
train = df[df["YEAR"] < 2025]
test = df[df["YEAR"] >= 2025]

# Definir las características a usar
features = [
    "TMAX",
    "TMIN",
    "TEMP_MEAN",
    "TEMP_RANGE",
    "SNOW"
]

# Preparar los conjuntos de entrenamiento y prueba
X_train = train[features]
y_train = train["RAIN"]

X_test = test[features]
y_test = test["RAIN"]

## 📌 Selección de variables (Feature Selection)

**Objetivo:**  
Reducir la dimensionalidad del dataset seleccionando las variables más relevantes para la predicción.

**Descripción:**  
Se utiliza el método `SelectKBest` con la función estadística `f_classif` (ANOVA F-test) para evaluar la relación entre cada variable predictora y la variable objetivo.

El procedimiento consiste en:

- Calcular la relevancia de cada feature en función de su capacidad para discriminar entre clases.
- Seleccionar las **k=3 variables más importantes** según este criterio.
- Transformar los conjuntos de entrenamiento y prueba para conservar únicamente dichas variables.

Finalmente, se identifican explícitamente las features seleccionadas.

**Resultados / Insights:**
- Se reduce la complejidad del modelo al trabajar con menos variables.
- Las features seleccionadas (`TMIN`, `TEMP_RANGE`, `SNOW`) presentan mayor poder predictivo respecto a la ocurrencia de lluvia.
- Este paso contribuye a mejorar la interpretabilidad y potencialmente el rendimiento del modelo.

In [24]:
from sklearn.feature_selection import SelectKBest, f_classif

# Selección de características usando SelectKBest con ANOVA F-value
selector = SelectKBest(score_func=f_classif, k=3)
X_train_selected = selector.fit_transform(X_train, y_train)
X_test_selected = selector.transform(X_test)

selected_features = [features[i] for i in selector.get_support(indices=True)]
print("Features seleccionadas:", selected_features)

Features seleccionadas: ['TMAX', 'TMIN', 'TEMP_RANGE']


## 📌 Entrenamiento de modelos de clasificación

**Objetivo:**  
Entrenar distintos modelos de clasificación sobre el dataset procesado para comparar su desempeño en la predicción de lluvia.

**Descripción:**  
En este bloque se entrenan tres algoritmos de clasificación:

- **Decision Tree Classifier**
- **Logistic Regression**
- **K-Nearest Neighbors (KNN)**

Previo al entrenamiento, se aplica una **estandarización de variables** mediante `StandardScaler`, transformando los datos para que tengan media 0 y desviación estándar 1.

Si bien algunos modelos como Decision Tree no requieren este tipo de escalado, se decide aplicarlo de forma uniforme a todos los modelos con el objetivo de:

- Mantener consistencia en el flujo de datos
- Simplificar la implementación de futuros pipelines
- Facilitar la incorporación de modelos más complejos en etapas posteriores del proyecto

Los modelos se entrenan utilizando el conjunto de entrenamiento (`X_train`, `y_train`), ya sea en su versión escalada o no según corresponda.

**Configuración de modelos:**

- **Decision Tree:**
  - Profundidad máxima limitada (`max_depth=4`) para evitar sobreajuste
  - Mínimo de muestras por hoja (`min_samples_leaf=20`) para reducir ruido

- **Logistic Regression:**
  - Número máximo de iteraciones aumentado (`max_iter=1000`) para asegurar convergencia

- **KNN:**
  - Número de vecinos definido en `k=5`, valor estándar para un equilibrio entre sesgo y varianza

**Resultados / Insights:**
- Se obtiene un conjunto de modelos entrenados listos para evaluación y comparación.
- La estandarización permite que los modelos basados en distancia (KNN) y optimización (Logistic Regression) funcionen correctamente.
- La inclusión de múltiples modelos permite analizar distintas aproximaciones al problema: reglas (árbol), linealidad (regresión logística) y proximidad (KNN).

In [28]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

# --- Estandarizado de escalado para modelos sensibles a la escala
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_selected)
X_test_scaled = scaler.transform(X_test_selected)

# --- Modelos a entrenar
log_reg = LogisticRegression(max_iter=1000)
knn = KNeighborsClassifier(n_neighbors=5)
tree = DecisionTreeClassifier( # No es necesario estandarizar para árboles, pero lo hacemos para mantener consistencia
    max_depth=4,
    min_samples_leaf=20,
    random_state=42
)

tree.fit(X_train_scaled, y_train)
log_reg.fit(X_train_scaled, y_train)
knn.fit(X_train_scaled, y_train)

,n_neighbors,5
,weights,'uniform'
,algorithm,'auto'
,leaf_size,30
,p,2
,metric,'minkowski'
,metric_params,None
,n_jobs,None


## 📌 Evaluación y comparación de modelos

**Objetivo:**  
Evaluar el desempeño de los distintos modelos de clasificación entrenados y compararlos para identificar cuál se ajusta mejor al problema de predicción de lluvia.

**Descripción:**  
Se evalúan los modelos utilizando las siguientes métricas:

- **Accuracy:** proporción de predicciones correctas
- **Matriz de confusión:** permite analizar errores por clase
- **Precision, Recall y F1-score:** métricas clave para evaluar el desempeño en problemas de clasificación

Cada modelo se evalúa sobre el conjunto de prueba, permitiendo comparar su capacidad de generalización.

---

## 🔍 Análisis de resultados

### 📈 Logistic Regression
- Mejor desempeño general en términos de accuracy
- Buen equilibrio entre precision y recall
- Mejor capacidad de generalización en este dataset

### 🌳 Decision Tree
- Buen desempeño general
- Mayor recall en la clase 0 (no lluvia)
- Menor capacidad para detectar correctamente eventos de lluvia (clase 1)

### 📍 KNN
- Desempeño similar al árbol de decisión
- Sensible a la distribución de los datos
- Menor estabilidad en comparación con Logistic Regression

In [34]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

models = {
    "Decision Tree": (tree, X_test_selected),
    "Logistic Regression": (log_reg, X_test_scaled),
    "KNN": (knn, X_test_scaled)
}

results = {}

# Evaluación de cada modelo
for name, (model, X_data) in models.items():
    
    y_pred = model.predict(X_data)
    
    acc = accuracy_score(y_test, y_pred)
    results[name] = acc
    
    print(f"\n====================")
    print(f"Modelo: {name}")
    print(f"Accuracy: {acc:.3f}")
    
    print("\nMatriz de confusión:")
    print(confusion_matrix(y_test, y_pred))
    
    print("\nReporte de clasificación:")
    print(classification_report(y_test, y_pred))


Modelo: Decision Tree
Accuracy: 0.660

Matriz de confusión:
[[160  32]
 [ 92  81]]

Reporte de clasificación:
              precision    recall  f1-score   support

           0       0.63      0.83      0.72       192
           1       0.72      0.47      0.57       173

    accuracy                           0.66       365
   macro avg       0.68      0.65      0.64       365
weighted avg       0.67      0.66      0.65       365


Modelo: Logistic Regression
Accuracy: 0.699

Matriz de confusión:
[[160  32]
 [ 78  95]]

Reporte de clasificación:
              precision    recall  f1-score   support

           0       0.67      0.83      0.74       192
           1       0.75      0.55      0.63       173

    accuracy                           0.70       365
   macro avg       0.71      0.69      0.69       365
weighted avg       0.71      0.70      0.69       365


Modelo: KNN
Accuracy: 0.663

Matriz de confusión:
[[151  41]
 [ 82  91]]

Reporte de clasificación:
              pre

## 📊 Resultados obtenidos

**Accuracy por modelo:**

- Logistic Regression: **0.70**
- KNN: **0.66**
- Decision Tree: **0.66**

---

## 🧠 Conclusiones

- El modelo que mejor se ajusta al problema es **Logistic Regression**, al presentar la mayor accuracy y un balance adecuado entre métricas.
- Los modelos basados en reglas (Decision Tree) y proximidad (KNN) muestran un desempeño aceptable, pero inferior.
- La predicción de la clase positiva (lluvia) presenta mayor dificultad en todos los modelos, evidenciado por valores más bajos de recall.
- La simplicidad y capacidad de generalización de Logistic Regression lo convierten en una opción adecuada para este problema.

---

## 🚀 Próximos pasos

- Implementar un pipeline de procesamiento y modelado
- Explorar técnicas de mejora como tuning de hiperparámetros
- Evaluar modelos más complejos en etapas posteriores (DS2 / DS3)

In [30]:
import pandas as pd

# Mostrar resultados ordenados por accuracy
pd.DataFrame(results, index=["Accuracy"]).T.sort_values(by="Accuracy", ascending=False)

,Accuracy
Logistic Regression,0.698630
KNN,0.663014
Decision Tree,0.660274


## 📌 Implementación de pipelines

Se implementan pipelines utilizando `sklearn.pipeline` con el objetivo de unificar el flujo de preprocesamiento y entrenamiento de modelos.

Esto permite:

- Estandarizar el tratamiento de los datos
- Reducir la repetición de código
- Facilitar la reutilización en futuras etapas del proyecto

Cada pipeline incluye:
- Un paso de escalado (`StandardScaler`)
- Un modelo de clasificación

Este enfoque simplifica la comparación entre modelos y prepara la base para futuras mejoras como optimización de hiperparámetros.

In [35]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Función para construir pipelines
def build_pipelines(models_dict):
    pipelines = {}

    for name, model in models_dict.items():
        pipelines[name] = Pipeline([
            ("scaler", StandardScaler()),
            ("model", model)
        ])
    
    return pipelines


# Definimos los modelos sin estandarizar, ya que el pipeline se encargará de eso
models = {
    "Decision Tree": DecisionTreeClassifier(max_depth=4, min_samples_leaf=20, random_state=42),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "KNN": KNeighborsClassifier(n_neighbors=5)
}

# Construimos los pipelines
pipelines = build_pipelines(models)